# BBDD ESTADISTICA JUDICIAL (Consejo General del Poder Judicial)

ChatGPT ha descubierto que el C.G.P.J. tiene una base de datos pública, que permite un acceso más sofisticado que la descarga de Excels y PDFs.

Permite generar un enlace permanenente a una tabla preconfigurada, para exportarla a una variedad de formatos.

A pesar de que tanto ChatGPT como Brave han mencionado algún tipo de API REST, esta BD no tiene API pública (que se haya podido confirmar). Se han hecho pruebas y, si bien hay una configuración accesible públicamente, los datos no se han podido conseguir a tavés de API REST. El ICANE sí que expone APIs, pero solo para datos relacionados con Cantabria.

In [ ]:
import pandas as pd
import requests

from io import BytesIO, StringIO
from pathlib import Path

PATH_DATA = Path("../data")

## Descarga automática

### Descarga a través de enlace pregenerado

Como alternativa menos automática a la API, se puede pregenerar un enlace de descarga a una tabla en concreto.

El problema es que esto se debe hacer cada año, pues no se pueden marcar los años que aún no están en los datos.

#### Generación manual del enlace

1. Dirigirse a https://www6.poderjudicial.es/PXWeb-2025-v1/pxweb/es/

2. Entrar a la base de datos que nos interese (por ejemplo [11.-Juzgados de Violencia Sobre La Mujer](https://www6.poderjudicial.es/PXWeb-2025-v1/pxweb/es/11.-Juzgados%20de%20Violencia%20Sobre%20La%20Mujer/)).

3. Entrar a la tabla que nos interese (por ejemplo [PERSONAS ENJUICIADAS.VIOLENCIA DE GENERO Y SEXUAL . Juzgados de Violencia sobre la Mujer](https://www6.poderjudicial.es/PXWeb-2025-v1/pxweb/es/11.-Juzgados%20de%20Violencia%20Sobre%20La%20Mujer/-/OUJVM007.px/))

- Nota: se puede dar directamente en enlace a una BD en particular o a una tabla en particular.

4. Hacer la selección que nos interese (con un máximo de 100.000 filas) de variables y valores. Por ejemplo:

    - ANNOS: 2025
    - ORGJUD: JVM nr. 1 BARCELONA
    - PROCEDIMIENTOS: VG. Numero personas, VG. Condenado. Español, VG. Condenado. Extranjero, VS. Numero personas, VS. Condenado. Español, VS. Condenado. Extranjero
    - SITUACION: VARONES, MUJERES, TOTAL

4. Hacer clic en "Continuar"

5. En el menú lateral, seleccionar "Guarde su recuperación" y seleccionar el formato que nos interese. Por ejemplo "Delimitado por comas con encabezamiento".

    - Alerta: no recomiendo usar JSON porque los valores de las variables son todo enteros. Por ejemplo, en vez de leer "JVM nr. 1 BARCELONA" se leería un número entero arbitrario. Es muy poco intuitivo, y se necesitiaría hacer el mapeo (~500 items) posteriormente.

    - Alerta: parece que el sistema puede llegar a exportar en un formato diferente al seleccionado. En este ejemplo, ha usado ";" en lugar de "," para el separador.

6. Hacer clic en "Terminar" y copiar en enlace: https://www6.poderjudicial.es/PXWeb-2025-v1/sq/99adac52-6ae0-467a-a939-0e5988075ab5

In [67]:
url_ejemplo = (
    "https://www6.poderjudicial.es/"
    "PXWeb-2025-v1/sq/"
    "99adac52-6ae0-467a-a939-0e5988075ab5"
)

r = requests.get(url_ejemplo)

print("Status:", r.status_code)
print("Content-Type:", r.headers.get("Content-Type"))
print("Content-Disposition:", r.headers.get("Content-Disposition"))
print(r.text[:1000])

Status: 200
Content-Type: application/octet-stream; charset=Windows-1252
Content-Disposition: attachment; filename=OUJVM007.csv
"PERSONAS ENJUICIADAS.VIOLENCIA DE GENERO Y SEXUAL . Juzgados de Violencia sobre la Mujer por ANNOS, ORGJUD, PROCEDIMIENTOS y SITUACION"

"ANNOS";"ORGJUD";"PROCEDIMIENTOS";"VARONES";"MUJERES";"TOTAL"
"2025";"JVM nr. 1 BARCELONA";"VG. Numero personas";17;0;17
"2025";"JVM nr. 1 BARCELONA";"VG. Condenado. Español";5;0;5
"2025";"JVM nr. 1 BARCELONA";"VG. Condenado. Extranjero";6;0;6
"2025";"JVM nr. 1 BARCELONA";"VS. Numero personas";0;0;0
"2025";"JVM nr. 1 BARCELONA";"VS. Condenado. Español";0;0;0
"2025";"JVM nr. 1 BARCELONA";"VS. Condenado. Extranjero";0;0;0



In [ ]:
df_ejemplo = pd.read_csv(
    BytesIO(r.content),
    sep=";",
    encoding="Windows-1252"
)

df_ejemplo.head()

PERSONAS ENJUICIADAS.VIOLENCIA DE GENERO Y SEXUAL . Juzgados de Violencia sobre la Mujer por ANNOS, ORGJUD, PROCEDIMIENTOS y SITUACION
ANNOS ORGJUD              PROCEDIMIENTOS            VARONES MUJERES                                              TOTAL                                                                                    
2025  JVM nr. 1 BARCELONA VG. Numero personas       17      0                                                       17                                                                                    
                          VG. Condenado. Español    5       0                                                        5                                                                                    
                          VG. Condenado. Extranjero 6       0                                                        6                                                                                    
                          VS. Numero personas       0       0                                                        0

A continuación un ejemplo con JSON en lugar de CSV, que no recomiendo.

In [61]:
url_ejemplo = (
    "https://www6.poderjudicial.es/"
    "PXWeb-2025-v1/sq/"
    "7322e54e-5e37-41f1-9ba6-8b651cf9e5bb"
)

r = requests.get(url_ejemplo)

print("Status:", r.status_code)
print("Content-Type:", r.headers.get("Content-Type"))
print("Content-Disposition:", r.headers.get("Content-Disposition"))
print(r.text[:1000])

json_ejemplo = r.json()

# Extraer los nombres de las columnas
columnas = [col["code"] for col in json_ejemplo["columns"]]

# Extraer los datos
datos = [
    fila["key"] + fila["values"]
    for fila in json_ejemplo["data"]
]

# Crear DataFrame
df_ejemplo = pd.DataFrame(
    datos,
    columns=columnas
)

df_ejemplo.head()

Status: 200
Content-Type: text/json
Content-Disposition: attachment; filename=OUJVM007.json
{"columns":[{"code":"ANNOS","text":"ANNOS","type":"d"},{"code":"ORGJUD","text":"ORGJUD","type":"d"},{"code":"PROCEDIMIENTOS","text":"PROCEDIMIENTOS","type":"d"},{"code":"SITUACION","text":"SITUACION","type":"d"},{"code":"PERSONAS ENJUICIADAS.VIOLENCIA DE GENERO Y SEXUAL . Juzgados de Violencia sobre la Mujer","text":"PERSONAS ENJUICIADAS.VIOLENCIA DE GENERO Y SEXUAL . Juzgados de Violencia sobre la Mujer","type":"c"}],"comments":[],"data":[{"key":["2025","09081153001","2","1"],"values":["5"]}],"metadata":[{"updated":"9999-12-31T22:59:59Z","label":"PERSONAS ENJUICIADAS.VIOLENCIA DE GENERO Y SEXUAL . Juzgados de Violencia sobre la Mujer por ANNOS, ORGJUD, PROCEDIMIENTOS y SITUACION"}]}


,ANNOS,ORGJUD,PROCEDIMIENTOS,SITUACION,PERSONAS ENJUICIADAS.VIOLENCIA DE GENERO Y SEXUAL . Juzgados de Violencia sobre la Mujer
0,2025,09081153001,2,1,5


### Intento fallido de descubrir una API REST útil expuesta

Tal y como se ve a continuación, se puede acceder a la configuración, pero a nada más (que haya podido descubrir).

In [4]:
urls = [
    "https://www6.poderjudicial.es/PxWeb2023v1/api/v1/es/?config",
    "https://www6.poderjudicial.es/PxWeb2023v1/api/v1/es/",
    "https://www6.poderjudicial.es/PxWeb2023v1/api/v1/es/11.-Juzgados%20de%20Violencia%20Sobre%20La%20Mujer/OUJVM007.px"
]

for url in urls:
    try:
        r = requests.get(url, timeout=20)
        print("\nURL:", url)
        print("STATUS:", r.status_code)
        print("CONTENT-TYPE:", r.headers.get("Content-Type"))
        print("BODY:", r.text[:1500])
    except Exception as e:
        print(type(e).__name__, e)


URL: https://www6.poderjudicial.es/PxWeb2023v1/api/v1/es/?config
STATUS: 200
CONTENT-TYPE: application/json; charset=utf-8
BODY: {"maxValues":1000,"maxCells":100000,"maxCalls":10,"timeWindow":10,"CORS":true}

URL: https://www6.poderjudicial.es/PxWeb2023v1/api/v1/es/
STATUS: 400
CONTENT-TYPE: text/html
BODY: Solicitud incorrecta

URL: https://www6.poderjudicial.es/PxWeb2023v1/api/v1/es/11.-Juzgados%20de%20Violencia%20Sobre%20La%20Mujer/OUJVM007.px
STATUS: 400
CONTENT-TYPE: text/html
BODY: Solicitud incorrecta


### API del ICANE (solo para Cantabria)

La IA de Brave ha descubierto que el [ICANE (Instituto Cantábrico de Estadística)](https://www.icane.es/) expone una API con algunos datos del Consejo.

Concretamente, ha dicho esto:

___

La Base de datos de la estadística judicial (PC-AXIS) del Consejo General del Poder Judicial (CGPJ) ofrece acceso a sus datos mediante una API pública para uso programático. 

Los endpoints de la API están alojados en el dominio https://www.icane.es/data/api/ y permiten acceder a las estadísticas en múltiples formatos estructurados:

- JSON: Formato estándar para integración web y aplicaciones.
- PC-Axis (.px): Formato nativo para uso con el software local. 
- Excel (.xls): Formato de hoja de cálculo. 
- SDMX (ZIP): Formato estándar para intercambio de estadísticas. 

Cómo acceder: La URL base es https://www.icane.es/data/api/. Los recursos específicos siguen la nomenclatura `estadisticas-judiciales-[tipo]-[jurisdiccion]-[estado]-[partido-judicial].[formato]`. Por ejemplo, para obtener sentencias en formato JSON:

https://www.icane.es/data/api/estadisticas-judiciales-sentencias-jurisdiccion-estado-partido-judicial.json

Para consultas interactivas o descargas manuales, el CGPJ mantiene la herramienta online PC-AXIS en su portal oficial, donde se pueden seleccionar variables y exportar los datos sin necesidad de usar la API directamente.
___

Tras hacer pruebas, se ve que es solo para datos de Cantabria.

In [28]:
tipo_ejemplo = "sentencias"
jurisdiccion_ejemplo = "jurisdiccion"
estado_ejemplo = "estado"
partido_judicial_ejemplo = "partido-judicial"
formato_ejemplo = "csv"


url_ejemplo = (
    "https://www.icane.es/data/api/"
    f"estadisticas-judiciales-{tipo_ejemplo}-{jurisdiccion_ejemplo}-{estado_ejemplo}-{partido_judicial_ejemplo}.{formato_ejemplo}"
)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": (
        "https://www.icane.es/data/"
        "estadisticas-judiciales-sentencias-estado-jurisdiccion"
    ),
}

response = requests.get(
    url_ejemplo,
    headers=headers,
    timeout=30
)

print("Status:", response.status_code)
print("Headers:", dict(response.headers))
print("Body:", response.text[:5000])

Status: 200
Headers: {'Server': 'nginx', 'Date': 'Thu, 05 Nov 2026 11:15:12 GMT', 'Content-Type': 'text/csv;charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'set-cookie': 'JSESSIONID=2C69C0EAFB1786DC7E833B043252548F; Path=/data; HttpOnly', 'access-control-allow-origin': '*', 'content-disposition': 'attachment; filename= "Número de sentencias por jurisdicción, estado y partido judicial.csv"', 'Age': '0', 'Accept-Ranges': 'bytes', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'Expect-CT': 'enforce, max-age=43200', 'X-XSS-Protection': '1; mode=block', 'Referrer-Policy': 'no-referrer', 'Cache-Control': 'max-age=1'}
Body: "";"2006 - Castro-Urdiales - Registradas - Sentencias";"2006 - Castro-Urdiales - Resueltas - Sentencias";"2006 - Castro-Urdiales - En trámite al final del periodo - Sentencias";"2006 - Laredo - Registradas - Sentencias";"2006 - Laredo - Resueltas - Sentencias";"2006 - Laredo - En trámite al final del periodo - Sente

In [37]:
df_ejemplo = pd.read_csv(StringIO(response.text), sep=";")
df_ejemplo.head()

,Unnamed: 0,2006 - Castro-Urdiales - Registradas - Sentencias,2006 - Castro-Urdiales - Resueltas - Sentencias,2006 - Castro-Urdiales - En trámite al final del periodo - Sentencias,2006 - Laredo - Registradas - Sentencias,2006 - Laredo - Resueltas - Sentencias,2006 - Laredo - En trámite al final del periodo - Sentencias,2006 - Medio Cudeyo - Registradas - Sentencias,2006 - Medio Cudeyo - Resueltas - Sentencias,2006 - Medio Cudeyo - En trámite al final del periodo - Sentencias,...,2025 - San Vicente de la Barquera - En trámite al final del periodo - Sentencias,2025 - Santander - Registradas - Sentencias,2025 - Santander - Resueltas - Sentencias,2025 - Santander - En trámite al final del periodo - Sentencias,2025 - Santoña - Registradas - Sentencias,2025 - Santoña - Resueltas - Sentencias,2025 - Santoña - En trámite al final del periodo - Sentencias,2025 - Torrelavega - Registradas - Sentencias,2025 - Torrelavega - Resueltas - Sentencias,2025 - Torrelavega - En trámite al final del periodo - Sentencias
0,Total,466.0,365.0,808.0,432.0,431.0,486.0,424.0,335.0,824.0,...,600.0,6401.0,7175.0,14199.0,734.0,247.0,2402.0,1940.0,2339.0,6219.0
1,Jurisdicción Civil,296.0,223.0,630.0,303.0,302.0,423.0,216.0,143.0,739.0,...,490.0,5326.0,5927.0,13748.0,679.0,220.0,2256.0,1678.0,2026.0,5968.0
2,Jurisdicción Civil - Juzgados 1ª Instancia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,5010.0,5560.0,13008.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Jurisdicción Civil - Juzgados violencia contra...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,69.0,36.0,115.0,0.0,0.0,8.0,1.0,2.0,22.0
4,Jurisdicción Civil - Juzgados Familia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,247.0,331.0,625.0,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
for i, col in enumerate(df_ejemplo.columns):
    print(f"{i}: {col}")

0: Unnamed: 0
1: 2006 - Castro-Urdiales - Registradas - Sentencias
2: 2006 - Castro-Urdiales - Resueltas - Sentencias
3: 2006 - Castro-Urdiales - En trámite al final del periodo - Sentencias
4: 2006 - Laredo - Registradas - Sentencias
5: 2006 - Laredo - Resueltas - Sentencias
6: 2006 - Laredo - En trámite al final del periodo - Sentencias
7: 2006 - Medio Cudeyo - Registradas - Sentencias
8: 2006 - Medio Cudeyo - Resueltas - Sentencias
9: 2006 - Medio Cudeyo - En trámite al final del periodo - Sentencias
10: 2006 - Reinosa - Registradas - Sentencias
11: 2006 - Reinosa - Resueltas - Sentencias
12: 2006 - Reinosa - En trámite al final del periodo - Sentencias
13: 2006 - San Vicente de la Barquera - Registradas - Sentencias
14: 2006 - San Vicente de la Barquera - Resueltas - Sentencias
15: 2006 - San Vicente de la Barquera - En trámite al final del periodo - Sentencias
16: 2006 - Santander - Registradas - Sentencias
17: 2006 - Santander - Resueltas - Sentencias
18: 2006 - Santander - En tr

### Pendiente: intetar usar scraping para generar en enlace automáticamente